### Data Ingestion

In [17]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

from pathlib import Path

In [18]:
# read all pdf files
def process_all_pdfs(pdf_dir):
    '''Process all pdf files in a directory'''

    all_documents= []
    pdf_dir = Path(pdf_dir)

    # find all pdf files
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f'Total {len(pdf_files)} Pdf Files in Directory')

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader= PyPDFLoader(str(pdf_file))
            documents=loader.load()

            # add some metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" Error: {e}")
    print(f"Total {len(all_documents)} Documents Loaded.")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Total 3 Pdf Files in Directory
Processing: 3712001.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(34, 0, 1388021686480), '/BaseFont': '/LHCIMD+txsya', '/Encoding': IndirectObject(33, 0, 1388021686480), '/Widths': [744], '/FirstChar': 2, '/Type': '/Font', '/LastChar': 2}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Subtype': '/Type1', '/FontDescriptor': IndirectObject(773, 0, 1388021686480), '/BaseFont': '/LHDPGK+LinLibertineB', '/Encoding': '/WinAnsiEncoding', '/Widths': [256, 0, 0, 0, 0, 0, 0, 740, 654, 0, 734, 604, 0, 755, 817, 367, 373, 0, 577, 899, 0, 0, 614, 0, 0, 504, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 506, 542, 456, 561, 484, 391, 521, 619, 322, 312, 613, 325, 905, 616, 551, 581, 0, 428, 427, 358, 598, 529, 0, 561, 558], '/FirstChar': 58, '/Type': '/Font', '/LastChar': 121}, but is not installe

Loaded 39 pages
Processing: random_report_1.pdf
Loaded 1 pages
Processing: random_report_2.pdf
Loaded 1 pages
Total 41 Documents Loaded.


In [19]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'LaTeX with hyperref package', 'creationdate': '2025-02-07T10:06:31+05:30', 'subject': '', 'moddate': '2025-02-07T10:07:13+05:30', 'author': '', 'title': '', 'keywords': '', 'source': '..\\data\\pdf\\3712001.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'source_file': '3712001.pdf', 'file_type': 'pdf'}, page_content='Security and Privacy Challenges of Large Language Models:\nA Survey\nBADHAN CHANDRA DAS, Knight Foundation School of Computing and Information Sciences (KFS-\nCIS), Florida International University, Miami, United States\nM. HADI AMINI, Knight Foundation School of Computing and Information Sciences (KFSCIS), Florida\nInternational University, Miami, United States\nYANZHAO WU, Knight Foundation School of Computing and Information Sciences (KFSCIS), Florida\nInternational University, Miami, United States\nLarge language models (LLMs) have demonstrated extraordinary capabilities and contrib

### Splitting Documents into chunks

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [21]:
# split documents 

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    '''PSlit Documents into smaller chunksfor better RAG performance'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"Successfully split {len(documents)} into {len(split_docs)} chunks")

    return split_docs

In [22]:
chunks= split_documents(all_pdf_documents)
chunks

Successfully split 41 into 236 chunks


[Document(metadata={'producer': 'Acrobat Distiller 11.0 (Windows)', 'creator': 'LaTeX with hyperref package', 'creationdate': '2025-02-07T10:06:31+05:30', 'subject': '', 'moddate': '2025-02-07T10:07:13+05:30', 'author': '', 'title': '', 'keywords': '', 'source': '..\\data\\pdf\\3712001.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'source_file': '3712001.pdf', 'file_type': 'pdf'}, page_content='Security and Privacy Challenges of Large Language Models:\nA Survey\nBADHAN CHANDRA DAS, Knight Foundation School of Computing and Information Sciences (KFS-\nCIS), Florida International University, Miami, United States\nM. HADI AMINI, Knight Foundation School of Computing and Information Sciences (KFSCIS), Florida\nInternational University, Miami, United States\nYANZHAO WU, Knight Foundation School of Computing and Information Sciences (KFSCIS), Florida\nInternational University, Miami, United States\nLarge language models (LLMs) have demonstrated extraordinary capabilities and contrib

### Embeddings and VectoreDB

In [23]:
import chromadb
from chromadb.config import Settings
import numpy as np
from sentence_transformers import SentenceTransformer
import uuid
from typing import List, Tuple, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class EmbeddingsManager:
    '''This class handles embedding generation using Sentence Transformers'''

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        '''initialize the embedding manager
            model)name -> hugging face model name for sentence embeddings
        '''

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load sentence tranformer model"""
        try:
            print("Loading embedding model: ")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading Model -> {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if self.model is None:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        try:
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings for {len(texts)} texts. Embedding shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings -> {e}")
            raise

# initialize the embedding manager
embedding_manager = EmbeddingsManager()
embedding_manager

Loading embedding model: 


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1136.90it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [25]:
class VectorStore:
    '''Manages document embeddings in a ChromaDB vector store'''

    def __init__(self, persist_directory: str = "../data/chroma_db", collection_name: str = "pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # create persist Chroma Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, 
                                                                   
                metadata={"description": "Collection of PDF document embeddings"})

            print(f"ChromaDB collection '{self.collection_name}' initialized at '{self.persist_directory}'")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB -> {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store"""

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")

        try:
            # Prepare data for insertion
            ids = [str(uuid.uuid4()) for _ in range(len(documents))]
            metadatas = [doc.metadata for doc in documents]
            texts = [doc.page_content for doc in documents]

            # Add to collection
            self.collection.add(
                ids=ids,
                embeddings=embeddings.tolist(),
                metadatas=metadatas,
                documents=texts
            )

            print(f"Added {len(documents)} documents to the vector store.")
        except Exception as e:
            print(f"Error adding documents to ChromaDB -> {e}")
            raise

# initialize the vector store
vector_store = VectorStore()
vector_store

ChromaDB collection 'pdf_documents' initialized at '../data/chroma_db'
Existing documents in collection: 8


In [26]:
# convert text to embeddings
texts = [doc.page_content for doc in chunks]

# generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# store in vector store
vector_store.add_documents(chunks, embeddings)

Batches: 100%|██████████| 8/8 [00:21<00:00,  2.66s/it]


Generated embeddings for 236 texts. Embedding shape: (236, 384)
Added 236 documents to the vector store.


### Retrieving Query from RAG

In [29]:
class RAGRetriever:
    '''retriver for RAG using ChromaDB'''

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingsManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve the top_k nearest documents, optionally filtering by relevance score."""

        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and threshold={threshold}")

        try:
            # Generate embedding for the query
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]

            # Query the vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Prepare results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                    # Chroma's default L2 distance may exceed 1, so 1 - distance can be negative.
                    similarity_score = 1 / (1 + distance)
                    
                    if similarity_score >= threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents above the threshold of {threshold}.")
            
            else:
                print("No documents retrieved.")

            return retrieved_docs 
                
                
        except Exception as e:
            print(f"Error retrieving documents -> {e}")
            return []

In [30]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)

rag_retriever.retrieve("What is the main topic of the document?")

Retrieving documents for query: 'What is the main topic of the document?' with top_k=5 and threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.46it/s]

Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retrieved 5 documents above the threshold of 0.0.


[{'id': 'e67816ff-ba87-4c1b-8abc-f000d1daae2e',
  'content': 'The model is pre-trained with a large dataset containing public Internet data, books, and various\ntextstolearnalanguage’sunderlyingstructures,patterns,andcontextualrelationships.Thefoun-\ndationalcomponentofLLMs,sharedbynumerousLLMs,includingGPT-3[ 58],InstructGPT[ 150],\nLlama3[ 50],andGPT-4[ 3],istheself-attentionmechanism[ 206]intheTransformerarchitecture.\nIt plays a major role in the landscape of NLP by efficiently managing sequential data, facilitating\nparallelization, and capturing long-range dependencies in text data. During pre-training, the\ninput text is tokenized and processed by the model through the Transformer architecture. After\npre-training, the model undergoes a fine-tuning process for specific tasks or domains to enhance\nits performance for targeted applications. The model then generates output (e.g., next-word\nprediction or generating the sequence of words) based on the probability distributions of c

In [31]:
rag_retriever.retrieve("What is prompt Hacking?")

Retrieving documents for query: 'What is prompt Hacking?' with top_k=5 and threshold=0.0


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.98it/s]

Generated embeddings for 1 texts. Embedding shape: (1, 384)
Retrieved 5 documents above the threshold of 0.0.


[{'id': '586a24e8-d0ac-4f33-9122-2422ab1dfd9a',
  'content': 'need to put specific queries to the prompt, then it generates outputs based on training. Prompt\nhacking manipulates the input prompts of the LLM to obtain the desired output by a malicious\nuser. With proper malicious prompts, even a well-trained model can produce misleading or\nmalicious content [140]. The two types of goal-based prompt hacking strategies are described\nnext.\n4.1.1 Prompt Injection. Prompt injection entails bypassing LLM safety filters by manipulating\nthe model with meticulously crafted prompts that cause the model to disregard the original\nprompt. It may cause unintended consequences, such as data leakage, unauthorized access, and\nhate speech generation [173]. Recent studies explored several prompt injection attacks in LLMs.\nA straightforward way to mislead an LLM is to instruct it to ignore the previous prompt. This\nmethod is a combination ofgoal hijackingand prompt leaking[156]. Goal hijacking (al